# 08 — Chuẩn bị và xử lý chuyến đo

Nhập dữ liệu, kiểm tra GPS, ghép vào tuyến có hướng và tính chỉ tiêu vận hành. Đầu vào thực trống sẽ báo **chờ đo**. Chu trình lái nằm ngoài phạm vi notebook này.

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import display, Markdown

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / 'work' for p in candidates]
candidates += [Path.home() / 'python' / 'work']
WORK = next((p.resolve() for p in candidates if (p / 'scripts/lez/trip_processing.py').is_file()), None)
if WORK is None:
    raise FileNotFoundError('Mở notebook trong project có scripts/lez/trip_processing.py.')
if str(WORK / 'scripts') not in sys.path:
    sys.path.insert(0, str(WORK / 'scripts'))
from lez import trip_processing as trips
from lez import field_workflow

def short_paths(value):
    if isinstance(value, dict):
        return {short_paths(k): short_paths(v) for k, v in value.items()}
    if isinstance(value, list):
        return [short_paths(v) for v in value]
    if isinstance(value, str) and value.startswith(str(WORK) + '/'):
        return 'work/' + value[len(str(WORK)) + 1:]
    return value

def show(value):
    print(json.dumps(short_paths(value), ensure_ascii=False, indent=2))

show({'project': 'work/', 'kernel': 'Python thường; hình học dùng worker PyQGIS đã đăng ký'})

**Biểu mẫu và phiên bản tuyến.** Chuẩn bị checklist, lịch đề xuất và tệp đầu vào có dòng tiêu đề. Quan sát đã nhập được giữ lại. `route_version` và `source_bundle_sha256` ngăn chuyến cũ bị hiểu thành hình học mới khi mã tuyến được dùng lại.

In [ ]:
field = field_workflow.prepare(WORK)
show({k: field[k] for k in ['status', 'directory', 'selected_routes', 'incoming_directory',
                            'measured_trips_created', 'field_observations_created']})
field_state = field_workflow.status(field['directory'])
show({k: field_state[k] for k in ['status', 'stored_observation_rows', 'selected_routes_pending',
                                  'G1_G4_G5_G6_unknown_counts', 'C6_unknown_count']})

**Kiểm tra đầu vào.** Mỗi dòng chuyến gắn với tuyến, ngày đo theo giờ Hà Nội, policy và phiên bản nguồn. Vùng nghiên cứu A/B/C và trạng thái LEZ tại ngày đo được lưu riêng; phân vùng chưa xác nhận việc thực thi chính sách.

In [ ]:
check = trips.preflight(WORK)
show({k: check[k] for k in ['status', 'route_count', 'trip_rows', 'point_rows',
                            'input_files', 'schema_errors', 'source_snapshot_utc']})
show({'status_previous_real_run': trips.status(WORK)['status']})
# Tra phiên bản đúng để điền biểu mẫu của chiến dịch hiện tại.
first_route = sorted(check['route_contracts'])[0]
show({'route_id': first_route, **check['route_contracts'][first_route]})

**Chạy dữ liệu thực.** Lưu nguyên byte nguồn, tạo phiên bản xử lý riêng và gắn cờ từng lỗi. Với đầu vào trống, lệnh dừng ở `awaiting_measurements`; không tạo GPS hay thông số thực địa.

In [ ]:
real = trips.run(WORK)
show({k: real[k] for k in ['status', 'is_test_data', 'measurement_verified',
                           'trip_count', 'point_count', 'network_calls', 'driving_cycle_constructed']})
if 'point_flag_counts' in real:
    show({'point_flag_counts': real['point_flag_counts'],
          'trip_flag_counts': real['trip_flag_counts'],
          'map_matching': real['map_matching']['statuses'],
          'directory': real['directory']})

**Đọc chỉ tiêu cùng mẫu số.** Tốc độ trung bình dùng tích phân tốc độ chia thời gian hợp lệ. Thời gian idle/moving và tăng tốc/giảm tốc/cruise chỉ tính ở khoảng đủ dữ liệu; khoảng trống không được nội suy. Khoảng cách GPS, tích phân tốc độ và ghép tuyến được xuất riêng.

In [ ]:
if real.get('directory'):
    metrics = json.loads((Path(real['directory']) / 'operating_metrics.json').read_text())
    columns = ['trip_id', 'route_id', 'metadata_valid', 'point_count', 'declared_duration_s',
               'speed_valid_duration_s', 'speed_duration_coverage_fraction',
               'time_weighted_mean_speed_kmh', 'gps_distance_m', 'matched_distance_m',
               'idle_time_s', 'moving_time_s', 'flagged_point_count']
    show([{key: row[key] for key in columns} for row in metrics])
    aggregate_path = Path(real["directory"]) / "operating_summary.json"
    if aggregate_path.is_file():
        show(json.loads(aggregate_path.read_text()))
else:
    print('Chưa có chỉ tiêu vận hành từ dữ liệu thực. Chờ trips.csv và gps_points.csv.')


**Demo tùy chọn — dữ liệu kiểm thử riêng.** Mặc định tắt. Khi bật, sinh 31 điểm trên hình học tuyến hiện tại với tốc độ tùy ý để kiểm tra nhập/đổi hệ tọa độ/ghép có hướng. Demo lưu ở nhánh riêng với `is_test_data=true`; không đại diện giao thông Hà Nội.

In [ ]:
RUN_DEMO = False
if RUN_DEMO:
    fixture = trips.generate_demo(WORK)
    demo = trips.run(WORK, input_dir=fixture['directory'], demo=True)
    show({k: demo[k] for k in ['status', 'is_test_data', 'trip_count', 'point_count', 'directory']})
    show({'synthetic_match_statuses': demo['map_matching']['statuses']})
else:
    print('Demo đang tắt; không sinh dữ liệu kiểm thử.')

**Phần đang chờ.** Ranh giới và tuyến dùng đúng phiên bản nguồn hiện tại. Kiểm tra trên máy không điền G1/G4/G5/G6, C6 hay kết quả đo thực địa thay cho bằng chứng.

In [ ]:
pending = []
if real['trip_count'] == 0 or real['point_count'] == 0:
    pending.append('Chưa có bộ chuyến/GPS thực để đánh giá vận hành và chất lượng thiết bị.')
if field_state['selected_routes_pending']:
    pending.append(f"{field_state['selected_routes_pending']} tuyến còn chờ kiểm tra hiện trường và bằng chứng.")
if field_state['C6_unknown_count']:
    pending.append(f"C6 quan sát còn trống ở {field_state['C6_unknown_count']} tuyến.")
if check['schema_errors']:
    pending.append('Cần sửa schema CSV theo báo cáo; giữ bản nguồn nguyên byte.')
if real.get('trip_flag_counts'):
    pending.append('Một số chuyến có lỗi metadata/phiên bản; xem trips_qc.json trước khi dùng chỉ tiêu.')
if real.get('point_flag_counts'):
    pending.append('Một số điểm có cờ QC; xem points_qc.json và tỷ lệ thời gian hợp lệ.')
show({'pending': pending, 'surveyed_set_locked': field_state['surveyed_set_locked'],
      'measurement_verified': False, 'driving_cycle_constructed': False})